In [3]:
library(dplyr)
library(NADA2)


Attaching package: 'dplyr'


The following objects are masked from 'package:stats':

    filter, lag


The following objects are masked from 'package:base':

    intersect, setdiff, setequal, union


Loading required package: EnvStats


Attaching package: 'EnvStats'


The following objects are masked from 'package:stats':

    predict, predict.lm


The following object is masked from 'package:base':

    print.default




In [25]:
# Reload original data
df <- read.csv("C:/Users/leila/Dropbox/MayoWetlands/wetland_alldata_2025_only.csv")

# Separate upland and wetland sites
upland_sites <- df %>%
  filter(grepl("upland", SiteID, ignore.case = TRUE))

wetland_sites <- df %>%
  filter(!grepl("upland", SiteID, ignore.case = TRUE))

cat(sprintf("Upland sites: %d\n", nrow(upland_sites)))
cat(sprintf("Wetland sites: %d\n", nrow(wetland_sites)))

# Define chemistry columns
chem_cols <- names(wetland_sites)[which(names(wetland_sites) == "Al") : which(names(wetland_sites) == "SO4_mg.L")]

# Convert "<" to negative values for wetland sites only
for(col in chem_cols) {
  if(is.character(wetland_sites[[col]])) {
    wetland_sites[[col]] <- ifelse(grepl("<", wetland_sites[[col]]), 
                        -as.numeric(gsub("<", "", wetland_sites[[col]])),
                        as.numeric(wetland_sites[[col]]))
  }
}


Upland sites: 22
Wetland sites: 68


# NEW: Average Chem data adn Remove TDS

In [26]:
# ---- Remove TDS from dataset before chemistry averaging ----
if ("TDS_mg.L" %in% names(wetland_sites)) {
  wetland_sites <- wetland_sites %>% select(-TDS_mg.L)
  message("Removed TDS_mg.L from wetland_sites")
}

# Define the chem data 
chem_cols <- names(wetland_sites)[
  which(names(wetland_sites) == "Al") :
  which(names(wetland_sites) == "ORPmV")
]

message("Detected chemistry variables: ", length(chem_cols))
print(chem_cols)

# Average the raw chem per site 
wetland_sites <- wetland_sites %>%
  mutate(SiteGroup = substr(SiteID, 1, 3)) %>%   # BPR01 → BPR
  group_by(SiteGroup) %>%
  summarise(
    # chemistry variables → average across replicates
    across(all_of(chem_cols), ~ mean(.x, na.rm = TRUE)),
    
    # all non-chem variables → first value (identical across replicates)
    across(-all_of(chem_cols), ~ dplyr::first(.x)),
    
    .groups = "drop"
  ) %>%
  mutate(SiteID = SiteGroup)  # keep consistent site naming

message("Averaged chemistry per site. New size: ", nrow(wetland_sites), " sites.")


Your code contains a unicode char which cannot be displayed in your
current locale and R will silently convert it to an escaped form when the
R kernel executes this code. This can lead to subtle errors if you use
such chars to do comparisons. For more information, please see
https://github.com/IRkernel/repr/wiki/Problems-with-unicode-on-windows

Removed TDS_mg.L from wetland_sites

Detected chemistry variables: 55



 [1] "Al"                          "Sb"                         
 [3] "As"                          "Ba"                         
 [5] "Be"                          "Bi"                         
 [7] "B"                           "Cd"                         
 [9] "Ca"                          "Cs"                         
[11] "Cr"                          "Co"                         
[13] "Cu"                          "F"                          
[15] "Fe"                          "Pb"                         
[17] "Li"                          "Mg"                         
[19] "Mn"                          "Mo"                         
[21] "Ni"                          "P"                          
[23] "K"                           "Rb"                         
[25] "Se"                          "Si"                         
[27] "Ag"                          "Na"                         
[29] "Sr"                          "S"                          
[31] "Te"                

Averaged chemistry per site. New size: 27 sites.



# Custom Function

In [27]:
# Calculate censored proportions and drop highly censored variables
censored_proportions <- sapply(wetland_sites[chem_cols], function(x) {
  not_na <- !is.na(x)
  if(sum(not_na) > 0) {
    return(sum(x[not_na] < 0) / sum(not_na))
  }
  return(0)
})

cols_to_drop <- names(censored_proportions[censored_proportions >= 0.60])
cat(sprintf("\nDropping %d variables with ≥60%% censoring\n", length(cols_to_drop)))
if(length(cols_to_drop) > 0) {
  print(cols_to_drop)
}

# Drop highly censored columns from BOTH wetland and upland dataframes
if(length(cols_to_drop) > 0) {
  wetland_sites <- wetland_sites %>% select(-all_of(cols_to_drop))
  upland_sites <- upland_sites %>% select(-all_of(cols_to_drop))
}

chem_cols_retained <- names(censored_proportions[censored_proportions < 0.60])

# ============================================================
# U-score transformation for WETLAND sites only
# ============================================================
cat("\n=== CALCULATING U-SCORES FOR WETLAND SITES ===\n")

for(col in chem_cols_retained) {
  values <- wetland_sites[[col]]
  not_na <- !is.na(values)
  
  if(sum(not_na) > 0) {
    values_no_na <- values[not_na]
    n <- length(values_no_na)
    
    # Rank the values (handles censored by using absolute values)
    ranks <- rank(abs(values_no_na), ties.method = "average")
    
    # Convert ranks to normal quantiles (u-scores)
    uscore_values <- rep(NA, length(values))
    uscore_values[not_na] <- qnorm((ranks - 0.5) / n)
    
    wetland_sites[[col]] <- uscore_values
  }
}

# ============================================================
# Combine wetland and upland sites back together
# ============================================================
cat("\n=== COMBINING WETLAND AND UPLAND SITES ===\n")

df_combined <- bind_rows(wetland_sites, upland_sites)

cat(sprintf("Final dataset: %d rows\n", nrow(df_combined)))
cat(sprintf("  Wetland sites: %d\n", nrow(wetland_sites)))
cat(sprintf("  Upland sites: %d\n", nrow(upland_sites)))

# Verify results
cat("\n=== U-SCORE TRANSFORMATION COMPLETE ===\n")
cat("First 10 wetland rows of chemistry:\n")
print(wetland_sites[chem_cols_retained] %>% head(10))

cat("\nRanges for wetland u-scores (should be approximately -3 to +3):\n")
ranges <- sapply(wetland_sites[chem_cols_retained], function(x) 
  sprintf("%.2f to %.2f", min(x, na.rm=TRUE), max(x, na.rm=TRUE)))
print(ranges)

# Save the combined dataset
output_path <- "C:/Users/leila/Dropbox/MayoWetlands/wetland_alldata_2025_only_uscore.csv"
write.csv(df_combined, output_path, row.names = FALSE)

cat(sprintf("Saved to: %s\n", output_path))


Dropping 9 variables with =60% censoring
[1] "Be"  "Bi"  "Cs"  "Te"  "Tl"  "Th"  "W"   "NO3" "NO4"

=== CALCULATING U-SCORES FOR WETLAND SITES ===

=== COMBINING WETLAND AND UPLAND SITES ===
Final dataset: 49 rows
  Wetland sites: 27
  Upland sites: 22

=== U-SCORE TRANSFORMATION COMPLETE ===
First 10 wetland rows of chemistry:
           Al         Sb         As         Ba          B         Cd         Ca
1  -1.5547736 -0.7721932  1.5547736  0.5244005 -0.7721932 -0.9153651  1.5547736
2   0.9153651 -1.0803193  0.4124631  0.3054808  0.2018935 -0.6433454  0.0000000
3   0.3054808  0.4124631  0.2018935 -0.5244005 -1.7506861  0.0000000  0.5244005
4   0.5244005 -0.4124631 -0.6433454  0.7721932  0.2018935 -0.3054808  0.2018935
5   2.0537489  0.5244005 -0.2018935  2.0537489 -0.9153651  0.6433454 -0.5244005
6   1.2815516 -0.1004337  0.9153651  1.5547736 -1.0803193  0.4124631 -0.7721932
7   1.5547736  0.3054808 -0.4124631 -0.3054808  0.2018935  0.9153651 -0.6433454
8   0.6433454 -0.7721932  0.3

# Compare the custom function and NADA2 function

In [13]:
library(NADA2)
library(dplyr)


# Drop upland sites
df <- df %>%
  filter(!grepl("upland", SiteID, ignore.case = TRUE))

# Define chemistry columns
chem_cols <- names(df)[which(names(df) == "Al") : which(names(df) == "SO4_mg.L")]

# Convert "<" to negative values if needed
for(col in chem_cols) {
  if(is.character(df[[col]])) {
    df[[col]] <- ifelse(grepl("<", df[[col]]), 
                        -as.numeric(gsub("<", "", df[[col]])),
                        as.numeric(df[[col]]))
  }
}

# Test NADA2::uscore on Al column
cat("=== TESTING NADA2::uscore() ===\n")
test_vals <- df$Al[!is.na(df$Al)]

cat(sprintf("Total values: %d\n", length(test_vals)))
cat("First 20 values:\n")
print(head(test_vals, 20))

# Prepare for uscore
y_vals <- abs(test_vals)  # absolute values
cen_ind <- test_vals < 0   # censored indicator (TRUE if censored)

cat(sprintf("\nCensored: %d (%.1f%%)\n", sum(cen_ind), sum(cen_ind)/length(cen_ind)*100))

# Call NADA2::uscore with rnk = FALSE
uscores <- NADA2::uscore(y_vals, cen_ind, rnk = FALSE)

cat("\nU-scores from NADA2::uscore(rnk = FALSE):\n")
cat("First 20 u-scores:\n")
print(head(uscores, 20))

cat(sprintf("\nU-score range: %.3f to %.3f\n", min(uscores), max(uscores)))
cat(sprintf("U-score mean: %.3f\n", mean(uscores)))
cat(sprintf("U-score sd: %.3f\n", sd(uscores)))

# Check if they're continuous or integers
cat("\nAre u-scores integers?\n")
cat(sprintf("All integers: %s\n", all(uscores == round(uscores))))

# Also test with rnk = TRUE to compare
uscores_rank <- NADA2::uscore(y_vals, cen_ind, rnk = TRUE)
cat("\n=== COMPARISON: rnk = TRUE ===\n")
cat("First 20 values:\n")
print(head(uscores_rank, 20))
cat(sprintf("Range: %.3f to %.3f\n", min(uscores_rank), max(uscores_rank)))

=== TESTING NADA2::uscore() ===
Total values: 66
First 20 values:
 [1]  0.0011  0.0034 -0.0010  0.2360  0.1210  0.2300  0.0570  0.0791  0.0257
[10]  0.1850  0.0041  1.5200  2.2100  1.0200  0.0786  0.5850  0.5010  0.6360
[19]  0.5880  0.6030

Censored: 2 (3.0%)

U-scores from NADA2::uscore(rnk = FALSE):
First 20 u-scores:
 [1] -61 -45 -64  43  35  41  17  27   7  39 -41  63  65  61  25  51  47  57  53
[20]  55

U-score range: -64.000 to 65.000
U-score mean: 0.000
U-score sd: 38.391

Are u-scores integers?
All integers: TRUE

=== COMPARISON: rnk = TRUE ===
First 20 values:
 [1]  3.0 11.0  1.5 55.0 51.0 54.0 42.0 47.0 37.0 53.0 13.0 65.0 66.0 64.0 46.0
[16] 59.0 57.0 62.0 60.0 61.0
Range: 1.500 to 66.000
